# 定价与 Greeks（Step 1–3）

Black-Scholes 解析解与 Put-Call Parity；蒙特卡洛定价与方差缩减；有限差分 / pathwise / 似然比三种 Greeks 方法（含数字期权）。

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np

from bs import (
    delta,
    gamma,
    greeks_finite_diff,
    price,
    put_call_parity_residual,
    rho,
    theta,
    vega,
)
from mc_greeks import (
    analytical_greek,
    bs_digital_delta,
    compare_greek_methods,
    fd_bias_variance_scan,
    mc_greek,
    plot_fd_bias_variance,
    plot_greeks_methods,
)
from mc_pricing import (
    compare_variance_reduction,
    confidence_interval_coverage,
    mc_european_call,
    plot_mc_convergence,
)

FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)

S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.2

## Step 1：BS 解析解与 Greeks

In [ ]:
c = price(S, K, T, r, sigma, "call")
p = price(S, K, T, r, sigma, "put")
print(f"看涨 C = {c:.10f}")
print(f"看跌 P = {p:.10f}")
print(f"Put-Call Parity 残差 = {put_call_parity_residual(S, K, T, r, sigma):.2e}")

print("\n解析 Greeks:")
print(f"  Delta = {delta(S, K, T, r, sigma, 'call'):.10f}")
print(f"  Gamma = {gamma(S, K, T, r, sigma):.10f}")
print(f"  Vega  = {vega(S, K, T, r, sigma):.10f}")
print(f"  Theta = {theta(S, K, T, r, sigma, 'call'):.10f}")
print(f"  Rho   = {rho(S, K, T, r, sigma, 'call'):.10f}")

fd = greeks_finite_diff(S, K, T, r, sigma, "call", h_s=1e-4, h_sigma=1e-4)
print("\n解析 vs 有限差分（价格层面）:")
for key in ("delta", "gamma", "vega", "rho", "theta"):
    ana = {"delta": delta(S, K, T, r, sigma, "call"), "gamma": gamma(S, K, T, r, sigma),
           "vega": vega(S, K, T, r, sigma), "rho": rho(S, K, T, r, sigma, "call"),
           "theta": theta(S, K, T, r, sigma, "call")}[key]
    rel = abs(ana - fd[key]) / max(abs(ana), 1e-8)
    print(f"  {key:6s}  rel err = {rel:.2e}")

## Step 2：蒙特卡洛定价与方差缩减

In [ ]:
n_paths = 100_000
vr = compare_variance_reduction(S, K, T, r, sigma, n_paths=n_paths, seed=42)
print(f"BS 解析看涨价 = {c:.6f}\n")
for name, est in vr.items():
    vr_ratio = est.variance_reduction_ratio
    vr_str = f"{vr_ratio:.2f}" if vr_ratio is not None else "—"
    print(
        f"{name:16s}  mean={est.mean:.6f}  SE={est.se:.6f}  "
        f"95% CI=[{est.ci_low:.4f}, {est.ci_high:.4f}]  VR={vr_str}"
    )

cov = confidence_interval_coverage(
    S, K, T, r, sigma, n_paths=5_000, n_trials=500, method="plain", base_seed=100
)
print(f"\n500 次重复：95% 置信区间覆盖率 = {cov:.3f}")

slope = plot_mc_convergence(
    S=S, K=K, T=T, r=r, sigma=sigma,
    ns=(10**2, 10**3, 10**4, 10**5, 10**6),
    seed=42,
    out_path=str(FIG / "fig_mc_convergence.png"),
)
print(f"MC 标准误 log-log 斜率（理论 -0.5）: {slope:.3f}")

## Step 3：三种 Greeks 方法（欧式看涨 + 数字期权）

In [ ]:
n_greek = 200_000
for greek in ("delta", "vega"):
    true = analytical_greek(S, K, T, r, sigma, greek, "call")
    res = compare_greek_methods(
        S, K, T, r, sigma, n_paths=n_greek, greek=greek, seed=11 if greek == "delta" else 12
    )
    print(f"\n欧式看涨 {greek}，解析真值 = {true:.6f}")
    for method, est in res.items():
        print(f"  {method:18s}  {est.mean:.6f} ± {est.se:.6f}  (bias={est.mean - true:+.6f})")

plot_greeks_methods(
    S=S, K=K, T=T, r=r, sigma=sigma,
    n_paths=80_000,
    seed=1,
    out_path=str(FIG / "fig_greeks_methods.png"),
)
print("已写入 figures/fig_greeks_methods.png")

In [ ]:
true_d = bs_digital_delta(S, K, T, r, sigma)
n_digital = 200_000
print(f"数字期权 Delta 解析真值 = {true_d:.6f}\n")

pw = mc_greek(S, K, T, r, sigma, n_digital, greek="delta", method="pathwise", payoff="digital", seed=3)
lr = mc_greek(S, K, T, r, sigma, n_digital, greek="delta", method="likelihood_ratio", payoff="digital", seed=3)
fd01 = mc_greek(S, K, T, r, sigma, n_digital, greek="delta", method="finite_diff", payoff="digital", seed=3, h=1e-2)
fd10 = mc_greek(S, K, T, r, sigma, n_digital, greek="delta", method="finite_diff", payoff="digital", seed=3, h=1e-1)

print(f"pathwise           {pw.mean:.6f} ± {pw.se:.6f}  （几乎处处导数为 0，失效）")
print(f"likelihood_ratio   {lr.mean:.6f} ± {lr.se:.6f}")
print(f"finite_diff h=0.01 {fd01.mean:.6f} ± {fd01.se:.6f}")
print(f"finite_diff h=0.1  {fd10.mean:.6f} ± {fd10.se:.6f}")
print(f"\nFD(h=0.01) SE / LR SE ≈ {fd01.se / lr.se:.1f}x")

In [ ]:
hs = np.array([1e-1, 3e-2, 1e-2, 3e-3, 1e-3, 3e-4, 1e-4])
h_arr, bias, var = fd_bias_variance_scan(
    S, K, T, r, sigma, hs=hs, n_paths=20_000, n_batches=40, base_seed=42
)
print("有限差分 Delta（欧式看涨，公共随机数）步长扫描:")
for h, b, v in zip(h_arr, bias, var):
    print(f"  h={h:.1e}  |bias|={b:.2e}  var={v:.2e}")

plot_fd_bias_variance(
    S=S, K=K, T=T, r=r, sigma=sigma,
    hs=tuple(hs),
    n_paths=20_000,
    n_batches=40,
    seed=2,
    out_path=str(FIG / "fig_fd_bias_variance.png"),
)
print("\n已写入 figures/fig_fd_bias_variance.png")